In [ ]:
!pip install -q transformers pypdf accelerate

from transformers import AutoTokenizer, AutoModelForCausalLM
from pypdf import PdfReader
import torch, re, json
from pathlib import Path

In [ ]:
PDF_PATH = Path("RAG_base/rag_base.pdf")
OUT_PATH = Path("domain_qa.jsonl")

# === Load small multilingual model ===
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", torch_dtype=torch.float16)

# === PDF -> text ===
def pdf_to_text(path):
    reader = PdfReader(str(path))
    return "\n\n".join((p.extract_text() or "") for p in reader.pages)

def clean_text(t):
    t = re.sub(r"(\w)-\n(\w)", r"\1\2", t)
    t = re.sub(r"[ \t]*\n(?!\s*\n)", " ", t)
    return re.sub(r"\s+\n", "\n", t).strip()

text = clean_text(pdf_to_text(PDF_PATH))

# Split into chapters (simple heuristic)
chapters = re.split(r"(?i)\n\s*(?:Kapitel|Chapter)\s+\d+", text)
chapters = [c.strip() for c in chapters if len(c.split()) > 80]  # skip tiny bits

print("Detected chapters:", len(chapters))
